# Validate Knowledge Graph Repository

Simple check on kg repo capabilities:
- start with seed id for a given paragraph (mentions greek polis)
- find entities in that paragraph
- find 1st order hops from those entities
- create graph with pyvis

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

seed_id = "de5a571d-f42d-45b5-82af-7e4862e43221"

In [2]:
from history_book.database.config.database_config import WeaviateConfig
from history_book.database.repositories.book_repository import BookRepositoryManager

config = WeaviateConfig.from_environment()
mgr = BookRepositoryManager(config)

# Quick counts
print(f"KG Entities:       {mgr.kg_entities.count()}")
print(f"KG Relationships:  {mgr.kg_relationships.count()}")
print(f"KG Graphs:         {mgr.kg_graphs.count()}")

# What graphs do we have?
for g in mgr.kg_graphs.list_all():
    print(f"\n  {g.name} ({g.graph_type}): {g.entity_count} entities, {g.relationship_count} rels, chapters={g.book_chapters}")

KG Entities:       227
KG Relationships:  453
KG Graphs:         1

  book3_ch2-3 (book): 227 entities, 453 rels, chapters=['3:2', '3:3']


In [3]:
# Which graph has this paragraph? Use the merged book graph.
graph_name = "book3_ch2-3"

# Step 1: Find entities in the seed paragraph
seed_entities = mgr.kg_entities.find_by_paragraph(seed_id, graph_name)
print(f"Entities in paragraph {seed_id[:8]}...:")
for e in seed_entities:
    print(f"  {e.name} ({e.entity_type}) — aliases={e.aliases}, occurrences={e.occurrence_count}")

Entities in paragraph de5a571d...:
  Alexander (person) — aliases=['Alexander the Great', 'the Great'], occurrences=13
  Macedonian governors (person) — aliases=[], occurrences=1
  Athenian democrats (polity) — aliases=[], occurrences=1
  Greeks (ancient Greek civilization) (polity) — aliases=['Greek settlers', 'Greek thought', 'early Greece', 'this civilization', 'Greeks', 'Hellenes', 'Greek society', 'Greece', 'Greek-speakers', 'Mainland Greece'], occurrences=10
  Polis (polity) — aliases=['Greek states', 'Greek city-state', 'cities of Greece', 'city-state', 'Greek city-states', 'Greek cities', 'city-states'], occurrences=11


In [4]:
# Step 2: Find 1st-order hop relationships from seed entities
seed_entity_ids = [e.id for e in seed_entities]
rels = mgr.kg_relationships.find_by_entities(seed_entity_ids, graph_name)
print(f"Relationships involving seed entities: {len(rels)}")
for r in rels:
    print(f"  {r.source_entity_name} --[{r.relation_type}]--> {r.target_entity_name}  (temporal: {r.temporal_context or '-'})")

Relationships involving seed entities: 136
  Olympian games --[participated_in]--> Greeks  (temporal: 776 BC)
  Hellenes --[part_of]--> Aegean  (temporal: eighth century and onwards)
  Hellenes --[part_of]--> Greek peninsula  (temporal: eighth century and onwards)
  Minoan Crete --[influenced]--> Greek civilization  (temporal: -)
  Mycenae --[influenced]--> Greek civilization  (temporal: -)
  Greek civilization --[part_of]--> Aegean  (temporal: -)
  Greeks --[settled]--> Aegean  (temporal: -)
  Greek world --[colonized]--> Thrace  (temporal: by the end of the sixth century BC)
  Greek world --[colonized]--> Levant  (temporal: by the end of the sixth century BC)
  Greek world --[colonized]--> South Italy  (temporal: by the end of the sixth century BC)
  Greeks --[part_of]--> Boeotian  (temporal: -)
  Greeks --[part_of]--> Dorian  (temporal: -)
  Greeks --[part_of]--> Ionian  (temporal: -)
  Homer --[influenced]--> Greek cities  (temporal: -)
  Greek world --[colonized]--> Black Sea Gree

In [5]:
# Step 3: Collect all connected entities (1-hop neighbors)
neighbor_ids = set()
for r in rels:
    neighbor_ids.add(r.source_entity_id)
    neighbor_ids.add(r.target_entity_id)

# Fetch neighbor entities not already in seed set
new_ids = neighbor_ids - set(seed_entity_ids)
all_entities = {e.id: e for e in seed_entities}
for nid in new_ids:
    entity = mgr.kg_entities.get_by_id(nid)
    if entity:
        all_entities[nid] = entity

print(f"Seed entities: {len(seed_entities)}")
print(f"1-hop neighbors: {len(new_ids)}")
print(f"Total subgraph entities: {len(all_entities)}")

Seed entities: 5
1-hop neighbors: 76
Total subgraph entities: 81


In [ ]:
# Step 4: Visualize subgraph with PyVis
from pyvis.network import Network

TYPE_COLORS = {
    "person": "#4e79a7",
    "polity": "#f28e2b",
    "place": "#e15759",
    "event": "#76b7b2",
    "concept": "#59a14f",
}

net = Network(height="600px", width="100%", notebook=True, cdn_resources="in_line")
net.barnes_hut(gravity=-3000, spring_length=150)

# Add nodes — seed entities are larger
for eid, e in all_entities.items():
    is_seed = eid in seed_entity_ids
    color = TYPE_COLORS.get(e.entity_type, "#999999")
    net.add_node(
        eid,
        label=e.name,
        title=f"{e.name}\nType: {e.entity_type}\nAliases: {', '.join(e.aliases) if e.aliases else '-'}\nOccurrences: {e.occurrence_count}",
        color=color,
        size=25 if is_seed else 15,
        borderWidth=3 if is_seed else 1,
    )

# Add edges
for r in rels:
    if r.source_entity_id in all_entities and r.target_entity_id in all_entities:
        net.add_edge(
            r.source_entity_id,
            r.target_entity_id,
            title=f"{r.relation_type}\n{r.temporal_context or ''}",
            label=r.relation_type,
            font={"size": 9},
        )

net.show("kg_subgraph.html")
print(f"Graph saved to kg_subgraph.html — {len(all_entities)} nodes, {len(rels)} edges")